# Short Quiz and small agent pipeline project.

This project implements a single-agent smart assistant that:

- Understands user queries
- Routes queries based on intent
- Uses tools when required
- Returns structured JSON output

## Supported Tasks

1. Mathematical queries → Calculator Tool
2. Keyword extraction → Keyword Extractor Tool
3. General queries → Direct Response

## Bonus Features

- Improved intent routing
- Basic error handling
- Logging
- Input validation

In [1]:
import re
import ast
import operator
import logging
import json

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("SingleAgent")

In [4]:
# TOOL 1: Calculator

def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression safely.
    Supports +, -, *, /, //, %, ** and parentheses.
    """

    try:
        expression = expression.strip()

        if not expression:
            return "Error in calculation: Empty expression"

        allowed_operators = {
            ast.Add: operator.add,
            ast.Sub: operator.sub,
            ast.Mult: operator.mul,
            ast.Div: operator.truediv,
            ast.FloorDiv: operator.floordiv,
            ast.Mod: operator.mod,
            ast.Pow: operator.pow,
            ast.USub: operator.neg,
            ast.UAdd: operator.pos
        }

        def evaluate(node):

            if isinstance(node, ast.Constant):

                if isinstance(node.value, (int, float)):
                    return node.value

                raise ValueError("Invalid value")

            elif isinstance(node, ast.BinOp):

                left = evaluate(node.left)
                right = evaluate(node.right)

                operation = allowed_operators.get(type(node.op))

                if operation is None:
                    raise ValueError("Unsupported operator")

                return operation(left, right)

            elif isinstance(node, ast.UnaryOp):

                operand = evaluate(node.operand)

                operation = allowed_operators.get(type(node.op))

                if operation is None:
                    raise ValueError("Unsupported operator")

                return operation(operand)

            else:
                raise ValueError("Invalid mathematical expression")

        tree = ast.parse(expression, mode="eval")
        result = evaluate(tree.body)

        return str(result)

    except ZeroDivisionError:
        return "Error in calculation: Division by zero"

    except Exception as e:
        logger.error(f"Calculator error: {e}")
        return "Error in calculation"

In [5]:
print(calculator("20 + 5"))
print(calculator("10 * 5"))
print(calculator("(20 + 5) * 2"))
print(calculator("100 / 4"))

25
50
50
25.0


In [6]:
# TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """
    Extract keywords from text.
    """

    try:

        if not text or not text.strip():
            return []

        cleaned_text = re.sub(r"[^\w\s]", "", text)

        words = cleaned_text.split()

        keywords = []

        for word in words:

            word = word.lower()

            if len(word) > 4 and word not in keywords:
                keywords.append(word)

        return keywords[:5]

    except Exception as e:

        logger.error(f"Keyword extraction error: {e}")

        return []

In [7]:
text = "Artificial Intelligence is transforming industries"

print(extract_keywords(text))

['artificial', 'intelligence', 'transforming', 'industries']


In [8]:
# GENERAL RESPONSE

def general_response(query: str) -> str:
    """
    Generate a basic response for general queries.
    """

    query = query.strip()

    if not query:
        return "Please enter a valid query."

    return (
        f"Your query is a general question. "
        f"You asked: '{query}'"
    )

In [9]:
# Extract mathematical expression from the query

def extract_expression(query: str) -> str:
    """
    Extract the mathematical expression from a user query.
    """

    expression = re.sub(
        r"\b(calculate|calculation|compute|solve)\b",
        "",
        query,
        flags=re.IGNORECASE
    )

    expression = expression.replace("?", "").strip()

    return expression

In [10]:
# AGENT FUNCTION

def agent(query: str):

    try:

        if not isinstance(query, str):
            return {
                "type": "error",
                "result": "Query must be a string."
            }

        query = query.strip()

        if not query:
            return {
                "type": "error",
                "result": "Query cannot be empty."
            }

        query_lower = query.lower()

        logger.info(f"Received query: {query}")

        # Route to Calculator
        if "calculate" in query_lower:

            logger.info("Routing to Calculator Tool")

            expression = extract_expression(query)

            result = calculator(expression)

            if result.startswith("Error"):

                return {
                    "type": "error",
                    "result": result
                }

            return {
                "type": "calculation",
                "result": result
            }

        # Route to Keyword Extractor
        elif "keywords" in query_lower:

            logger.info("Routing to Keyword Extractor Tool")

            text = re.sub(
                r"\b(extract|keywords|from)\b",
                "",
                query,
                flags=re.IGNORECASE
            ).strip()

            keywords = extract_keywords(text)

            return {
                "type": "keywords",
                "result": keywords
            }

        # Route to General Response
        else:

            logger.info("Routing to General Response")

            result = general_response(query)

            return {
                "type": "general",
                "result": result
            }

    except Exception as e:

        logger.exception("Unexpected agent error")

        return {
            "type": "error",
            "result": "An unexpected error occurred."
        }

In [11]:
# TEST CASES

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:

    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['artificial', 'intelligence', 'transforming', 'industries']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "Your query is a general question. You asked: 'What is machine learning?'"}
--------------------------------------------------


In [12]:
# Display responses as JSON

for q in queries:

    response = agent(q)

    print("Query:", q)
    print(json.dumps(response, indent=4))
    print("-" * 50)

Query: Calculate 20 + 5
{
    "type": "calculation",
    "result": "25"
}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
{
    "type": "keywords",
    "result": [
        "artificial",
        "intelligence",
        "transforming",
        "industries"
    ]
}
--------------------------------------------------
Query: What is machine learning?
{
    "type": "general",
    "result": "Your query is a general question. You asked: 'What is machine learning?'"
}
--------------------------------------------------


In [13]:
# ADDITIONAL TEST CASES

additional_queries = [
    "Calculate 100 / 4",
    "Calculate (10 + 20) * 3",
    "Calculate 2 ** 5",
    "Extract keywords from Python programming language and artificial intelligence",
    "What is Java?",
    "Tell me about software engineering",
    "",
    "Calculate 10 / 0",
    "keywords"
]

for q in additional_queries:

    print("Query:", q)

    response = agent(q)

    print(json.dumps(response, indent=4))

    print("=" * 60)

Query: Calculate 100 / 4
{
    "type": "calculation",
    "result": "25.0"
}
Query: Calculate (10 + 20) * 3
{
    "type": "calculation",
    "result": "90"
}
Query: Calculate 2 ** 5
{
    "type": "calculation",
    "result": "32"
}
Query: Extract keywords from Python programming language and artificial intelligence
{
    "type": "keywords",
    "result": [
        "python",
        "programming",
        "language",
        "artificial",
        "intelligence"
    ]
}
Query: What is Java?
{
    "type": "general",
    "result": "Your query is a general question. You asked: 'What is Java?'"
}
Query: Tell me about software engineering
{
    "type": "general",
    "result": "Your query is a general question. You asked: 'Tell me about software engineering'"
}
Query: 
{
    "type": "error",
    "result": "Query cannot be empty."
}
Query: Calculate 10 / 0
{
    "type": "error",
    "result": "Error in calculation: Division by zero"
}
Query: keywords
{
    "type": "keywords",
    "result": []


In [14]:
# IMPROVED INTENT DETECTION

def detect_intent(query: str) -> str:

    query_lower = query.lower()

    calculation_words = [
        "calculate",
        "calculation",
        "compute",
        "solve"
    ]

    keyword_words = [
        "keywords",
        "keyword",
        "extract keywords"
    ]

    if any(word in query_lower for word in calculation_words):
        return "calculation"

    if any(word in query_lower for word in keyword_words):
        return "keywords"

    return "general"

In [21]:
# IMPROVED AGENT

def improved_agent(query: str):

    try:

        if not isinstance(query, str):

            return {
                "type": "error",
                "result": "Query must be a string."
            }

        query = query.strip()

        if not query:

            return {
                "type": "error",
                "result": "Query cannot be empty."
            }

        intent = detect_intent(query)

        logger.info(f"Detected intent: {intent}")

        # Calculation
        if intent == "calculation":

            expression = extract_expression(query)

            result = calculator(expression)

            if result.startswith("Error"):

                return {
                    "type": "error",
                    "result": result
                }

            return {
                "type": "calculation",
                "result": result
            }

        # Keywords
        elif intent == "keywords":

            text = re.sub(
                r"\b(extract|keywords|keyword|from)\b",
                "",
                query,
                flags=re.IGNORECASE
            ).strip()

            keywords = extract_keywords(text)

            return {
                "type": "keywords",
                "result": keywords
            }

        # General
        else:

            return {
                "type": "general",
                "result": general_response(query)
            }

    except Exception as e:

        logger.exception("Unexpected error")

        return {
            "type": "error",
            "result": "An unexpected error occurred."
        }

In [22]:
# TEST IMPROVED AGENT

test_queries = [
    "Calculate 20 + 5",
    "Compute 100 / 4",
    "Solve 10 * 10",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "Give me keywords from Python programming language",
    "What is machine learning?",
    "Tell me about Java"
]

for q in test_queries:

    print("Query:", q)

    response = improved_agent(q)

    print(json.dumps(response, indent=4))

    print("-" * 60)

Query: Calculate 20 + 5
{
    "type": "calculation",
    "result": "25"
}
------------------------------------------------------------
Query: Compute 100 / 4
{
    "type": "calculation",
    "result": "25.0"
}
------------------------------------------------------------
Query: Solve 10 * 10
{
    "type": "calculation",
    "result": "100"
}
------------------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
{
    "type": "keywords",
    "result": [
        "artificial",
        "intelligence",
        "transforming",
        "industries"
    ]
}
------------------------------------------------------------
Query: Give me keywords from Python programming language
{
    "type": "keywords",
    "result": [
        "python",
        "programming",
        "language"
    ]
}
------------------------------------------------------------
Query: What is machine learning?
{
    "type": "general",
    "result": "Your query is

In [23]:
# INTERACTIVE MODE

print("Single-Agent Smart Assistant")
print("Type 'exit' to stop.")
print("-" * 50)

while True:

    user_input = input("Enter query: ")

    if user_input.lower().strip() == "exit":

        print("Agent stopped.")
        break

    response = improved_agent(user_input)

    print("Response:")
    print(json.dumps(response, indent=4))

    print("-" * 50)

Single-Agent Smart Assistant
Type 'exit' to stop.
--------------------------------------------------
Enter query: Calculate 2 ** 5
Response:
{
    "type": "calculation",
    "result": "32"
}
--------------------------------------------------
Enter query: Calculate 2 ** 10
Response:
{
    "type": "calculation",
    "result": "1024"
}
--------------------------------------------------
Enter query: Calculate 5 + 5
Response:
{
    "type": "calculation",
    "result": "10"
}
--------------------------------------------------
Enter query: exit
Agent stopped.


# System Architecture

```text
User Query
    |
    v
+---------------------+
|    Single Agent     |
|   Intent Detection  |
+----------+----------+
           |
     +-----+-----+
     |     |     |
     v     v     v
Calculate Keywords General
     |     |     |
     v     v     v
Calculator Keyword Direct
   Tool      Tool   Response
     |       |        |
     +-------+--------+
             |
             v
      Structured JSON
             |
             v
            User

In [19]:
# FINAL DEMONSTRATION

final_queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

print("SINGLE-AGENT SMART ASSISTANT")
print("=" * 60)

for query in final_queries:

    print(f"\nUser: {query}")

    response = improved_agent(query)

    print("Agent:")
    print(json.dumps(response, indent=4))

print("\n" + "=" * 60)
print("Assignment completed successfully.")

SINGLE-AGENT SMART ASSISTANT

User: Calculate 20 + 5
Agent:
{
    "type": "calculation",
    "result": "25"
}

User: Extract keywords from Artificial Intelligence is transforming industries
Agent:
{
    "type": "keywords",
    "result": [
        "artificial",
        "intelligence",
        "transforming",
        "industries"
    ]
}

User: What is machine learning?
Agent:
{
    "type": "general",
    "result": "Your query is a general question. You asked: 'What is machine learning?'"
}

Assignment completed successfully.


---

Completed the Single-Agent Smart Assistant project with intent-based query routing, Calculator and Keyword Extraction tools, structured JSON responses, error handling, logging, and interactive mode. The project was implemented and tested in Google Colab using Python.

---

**Submitted By:**  
**Mahesh Shinde**  
**Sanjivani College of Engineering, Kopargaon**  
**Student ID: CT_CSI_DS_1141**